In [1]:
import os
%pwd

'/home/sheshpal/Documents/ml_llmops_project/Data_science[1]-end-to-end/Data-science-Production/research'

In [2]:
os.chdir("../")
%pwd

'/home/sheshpal/Documents/ml_llmops_project/Data_science[1]-end-to-end/Data-science-Production'

In [14]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig: # the name is same as the config yaml file
    root_dir: Path
    source_url: str
    local_data_file: Path
    unzip_dir: Path


In [15]:
from src.Datascience.constants import *
from src.Datascience.utils.common import read_yaml, create_directories

In [16]:
class ConfigurationManager:
    def __init__(
        self,
        config_file_path=CONFIG_FILE_PATH,
        params_file_path=PARAMS_FILE_PATH,
        schema_file_path=SCHEMA_FILE_PATH):

        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_url=config.source_url,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )

        return data_ingestion_config


In [17]:
import os
import urllib.request as request
import zipfile
from src.Datascience import logger


In [18]:
# update the data ingestion component

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    # downloading the data
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, header = request.urlretrieve(
                url = self.config.source_url,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} downloaded with following info:\n{header}")
        else:
            logger.info(f"File already exists")

    def extract_zip_file(self):
        """
        Zip_file_path: str
        Extract the zip file into the directory
        function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)


In [19]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e


[2026-10-20 11:35:46,394:INFO:common:31:yaml file: config/config.yaml loaded successfully]
[2026-10-20 11:35:46,398:INFO:common:31:yaml file: params.yaml loaded successfully]
[2026-10-20 11:35:46,402:INFO:common:31:yaml file: schema.yaml loaded successfully]
[2026-10-20 11:35:46,404:INFO:common:59:Directory created: artifacts]
[2026-10-20 11:35:46,407:INFO:common:59:Directory created: artifacts/data_ingestion]
[2026-10-20 11:35:46,409:INFO:2090239321:16:File already exists]
